In [2]:
import torch
import torchvision
from torch import nn
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
from torchinfo import summary
from PIL import Image

In [3]:
data_path = Path("")

image_path = data_path / "deniz_canlilari"
train_dir = image_path / "train"
test_dir = image_path / "test"

In [4]:
NUM_WORKERS = os.cpu_count()

def create_data_loaders(
        train_dir : Path,
        test_dir : Path,
        batch_size: int,
        shuffle: bool = True,
        transform: transforms.Compose = None,
):
    train = datasets.ImageFolder(train_dir, transform=transform)
    test = datasets.ImageFolder(test_dir, transform=transform)

    class_names = train.classes

    train_loader = DataLoader(train, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    return train_loader, test_loader, class_names


In [5]:
weights = torchvision.models.ConvNeXt_Large_Weights.DEFAULT

auto_transforms = weights.transforms()

In [6]:
train_dataloader, test_dataloader, class_names = create_data_loaders(train_dir=train_dir,
                                                        test_dir=test_dir,
                                                        batch_size=32,
                                                        shuffle=True,
                                                        transform=auto_transforms)

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
model = torchvision.models.convnext_large(weights=weights).to(device)

In [9]:
summary(model=model, input_size=(32, 3, 224, 224), col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #                   Trainable
ConvNeXt                                      [32, 3, 224, 224]         [32, 1000]                --                        True
├─Sequential: 1-1                             [32, 3, 224, 224]         [32, 1536, 7, 7]          --                        True
│    └─Conv2dNormActivation: 2-1              [32, 3, 224, 224]         [32, 192, 56, 56]         --                        True
│    │    └─Conv2d: 3-1                       [32, 3, 224, 224]         [32, 192, 56, 56]         9,408                     True
│    │    └─LayerNorm2d: 3-2                  [32, 192, 56, 56]         [32, 192, 56, 56]         384                       True
│    └─Sequential: 2-2                        [32, 192, 56, 56]         [32, 192, 56, 56]         --                        True
│    │    └─CNBlock: 3-3                      [32, 192, 56, 56]         [32, 192, 56, 56]   

In [10]:
for param in model.features.parameters():
    param.requires_grad = False

In [11]:
summary(model=model, input_size=(32, 3, 224, 224), col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #                   Trainable
ConvNeXt                                      [32, 3, 224, 224]         [32, 1000]                --                        Partial
├─Sequential: 1-1                             [32, 3, 224, 224]         [32, 1536, 7, 7]          --                        False
│    └─Conv2dNormActivation: 2-1              [32, 3, 224, 224]         [32, 192, 56, 56]         --                        False
│    │    └─Conv2d: 3-1                       [32, 3, 224, 224]         [32, 192, 56, 56]         (9,408)                   False
│    │    └─LayerNorm2d: 3-2                  [32, 192, 56, 56]         [32, 192, 56, 56]         (384)                     False
│    └─Sequential: 2-2                        [32, 192, 56, 56]         [32, 192, 56, 56]         --                        False
│    │    └─CNBlock: 3-3                      [32, 192, 56, 56]         [32, 192, 56

In [12]:
output_features = len(class_names)

model.classifier[2] = nn.Linear(in_features=1536, out_features=output_features, bias=True)
model = model.to(device)

In [13]:
summary(model=model, input_size=(32, 3, 224, 224), col_names=["input_size","output_size","num_params","trainable"])

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #                   Trainable
ConvNeXt                                      [32, 3, 224, 224]         [32, 23]                  --                        Partial
├─Sequential: 1-1                             [32, 3, 224, 224]         [32, 1536, 7, 7]          --                        False
│    └─Conv2dNormActivation: 2-1              [32, 3, 224, 224]         [32, 192, 56, 56]         --                        False
│    │    └─Conv2d: 3-1                       [32, 3, 224, 224]         [32, 192, 56, 56]         (9,408)                   False
│    │    └─LayerNorm2d: 3-2                  [32, 192, 56, 56]         [32, 192, 56, 56]         (384)                     False
│    └─Sequential: 2-2                        [32, 192, 56, 56]         [32, 192, 56, 56]         --                        False
│    │    └─CNBlock: 3-3                      [32, 192, 56, 56]         [32, 192, 56

In [14]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [15]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device):
    # Put model in train mode
    model.train()

    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0

    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):

        X,y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metrics across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc


def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device):
    # Put model in eval mode
    model.eval()

    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0

    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item() / len(test_pred_labels))

    # Adjust metrics to get average loss and accuracy per batch
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc


def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          device: torch.device,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5):
    # 2. Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
               }

    # 3. Loop through training and testing steps for a number of epochs
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device)
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)

        # 4. Print out what's happening
        print(
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        # 5. Update results dictionary
        # Ensure all data is moved to CPU and converted to float for storage
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

    # 6. Return the filled results at the end of the epochs
    return results

In [16]:
results = train(model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                device=device,
                loss_fn=loss_fn,
                epochs=10)

Epoch: 1 | train_loss: 0.7737 | train_acc: 0.8152 | test_loss: 0.4309 | test_acc: 0.8729
Epoch: 2 | train_loss: 0.4150 | train_acc: 0.8801 | test_loss: 0.3891 | test_acc: 0.8889
Epoch: 3 | train_loss: 0.3514 | train_acc: 0.8984 | test_loss: 0.3705 | test_acc: 0.8927
Epoch: 4 | train_loss: 0.3175 | train_acc: 0.9074 | test_loss: 0.3571 | test_acc: 0.8995
Epoch: 5 | train_loss: 0.2952 | train_acc: 0.9145 | test_loss: 0.3600 | test_acc: 0.8952
Epoch: 6 | train_loss: 0.2698 | train_acc: 0.9237 | test_loss: 0.3637 | test_acc: 0.8982
Epoch: 7 | train_loss: 0.2562 | train_acc: 0.9243 | test_loss: 0.3580 | test_acc: 0.8998
Epoch: 8 | train_loss: 0.2418 | train_acc: 0.9303 | test_loss: 0.3575 | test_acc: 0.8984
Epoch: 9 | train_loss: 0.2298 | train_acc: 0.9305 | test_loss: 0.3552 | test_acc: 0.9031
Epoch: 10 | train_loss: 0.2163 | train_acc: 0.9342 | test_loss: 0.3559 | test_acc: 0.9032


In [20]:
img_tensor = auto_transforms(Image.open("dolphin.jpg")).unsqueeze(0).to(device)
predict = model(img_tensor).argmax(dim=1).item()

print(f"Tahmin edilen sınıf: {class_names[predict]}

Tahmin edilen sınıf: Dolphin | Gerçek sınıf: Dophin
